# I2C device initialization

In [1]:
%matplotlib tk
from equipment.equipment_init import *
from project.sc85853 import project
from interface.docs.output_chart import plot
import numpy as np
import math
from time import sleep as sleep


chart = plot()

if "ic" in globals():
    ic.close()
    
ic = project(device="sc85853", revision="0p0", emulator="cp2112", logging=False, is_gui=False)

initialized the dm6500 connection
initialized the keysight n6705 connection
initialized the scope connection
failed to initialize asd-906b
failed to initialize it8511a
initialized with i2c address 0x6e by default


In [5]:
ic.i2c_scan(update=1)

acked address list : [111]
update i2c address : 0x6e --> 0x6f


[111]

# Check Switching

In [14]:
bs.vset = 4
bs.enable

In [15]:
ps.ch2.cfg_all = 16.4, 3
ps.ch2.enable

In [17]:
ic.C1P2OUT_UVP=2
ic.NTC_FLT_DIS = 1
ic.NTC_ADC_DIS = 1
ic.VUSB_OVP = 7
ic.VUSB_SW_CTRL1 = 1
ic.IIN_UCP_DIS = 1
ic.VBAT_REG_DIS = 1
ic.IIN_REG_DIS = 1
print(ic.NTC_FLT_DIS)
ic.STANDBY_MODE_SET = 1
ic.QB1_CTRL2 = 1
ic.CP_EN=1

1


In [6]:

# ic.VEXT_OFF_GATE_CTRL = 1
print(ic.VEXT_OFF_GATE_CTRL)
print(ic.VUSB_OFF_GATE_CTRL)



0
0


In [8]:
ic.status


Addr    Reg           Value    Bit7                 Bit6                 Bit5                 Bit4                  Bit3                   Bit2                      Bit1                 Bit0
------  ------------  -------  -------------------  -------------------  -------------------  --------------------  ---------------------  ------------------------  -------------------  ---------------------
0x01    INT_DEVICE0   0x00     POR_FLAG             CP_SWITCHING_FLAG    VIN_IN_PRESENT_FLAG  VB_OUT_PRESENT_FLAG   VIN_IN_TH_CHG_EN_FLAG  VB_OUT_IN_TH_CHG_EN_FLAG  RSVD                 RSVD
0x02    INT_DEVICE1   0x00     VOUT_INSERT_FLAG     VOUT_TH_REV_EN_FLAG  VOUT_TH_CHG_EN_FLAG  VUSB_REMOVE_FLAG      VEXT_REMOVE_FLAG       VUSB_INSERT_FLAG          VEXT_INSERT_FLAG     RSVD
0x03    INT_DEVICE2   0x00     VUSB_OVP_FLAG        VUSB_DRV_ON_FLAG     VEXT_OVP_FLAG        VEXT_DRV_ON_FLAG      QB1_ON_FLAG            QB2_ON_FLAG               RSVD                 RSVD
0x04    INT_DEVICE3   0x00  

In [12]:
ic.CP_EN = 0
delay(3)
ps.ch2.disable
bs.disable

# Test code block

In [4]:
sleep(0.5)

# High Voltage Pins (VUSB ~ PMID)

In [12]:
# absolute maximum rating

#-------------------#
max_rating = 15
item = "VO1"
#-------------------#

temp = list(np.arange(0 , max_rating+10.2, 0.1))
voltage_sweep = [round(num, 1) for num in temp]
print(voltage_sweep)
ps.ch1.cfg_all = 0, 0.01
ps.ch2.cfg_all = 0, 0.01

chart.set_title = item

chart.set_single_plot
chart.set_xlabel = "Voltage (V)"
chart.set_y_main_label = "Current (uA)"

chart.add_main_item = "PIN"

chart.PIN.set_linewidth = 2
chart.PIN.set_line_dot

[np.float64(0.0), np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0), np.float64(1.1), np.float64(1.2), np.float64(1.3), np.float64(1.4), np.float64(1.5), np.float64(1.6), np.float64(1.7), np.float64(1.8), np.float64(1.9), np.float64(2.0), np.float64(2.1), np.float64(2.2), np.float64(2.3), np.float64(2.4), np.float64(2.5), np.float64(2.6), np.float64(2.7), np.float64(2.8), np.float64(2.9), np.float64(3.0), np.float64(3.1), np.float64(3.2), np.float64(3.3), np.float64(3.4), np.float64(3.5), np.float64(3.6), np.float64(3.7), np.float64(3.8), np.float64(3.9), np.float64(4.0), np.float64(4.1), np.float64(4.2), np.float64(4.3), np.float64(4.4), np.float64(4.5), np.float64(4.6), np.float64(4.7), np.float64(4.8), np.float64(4.9), np.float64(5.0), np.float64(5.1), np.float64(5.2), np.float64(5.3), np.float64(5.4), np.float64(5.5), np.float64(5.6), np.float64(5.7), np.float64(5.

In [13]:
ps.ch1.enable
delay(0.5)

for _ in range(3):
    dm1.current_100E_3

print("Voltage (V)   Current (uA)")

for set_volt in voltage_sweep:

    ps.ch1.vset = set_volt
    meas_i = dm1.current_100E_3 * 1e+6

    if meas_i > 2500:
        for _ in range(3):
            print("over-current, retest")
            meas_i = dm1.current_100E_3 * 1e+6
        
        if meas_i > 2500:
            break
    sleep(0.3)        

    chart.PIN.set_data = set_volt, meas_i
    chart.output

    print(f"{set_volt:1.01f}           {meas_i:.06f}")
    
chart.save_plot = chart.set_title
print(f"BV (V)           {set_volt:.01f}")

# rev_temp = list(np.arange(0, set_volt, 0.1))
# rev_voltage_sweep = [round(num, 1) for num in rev_temp]

# for rev in reversed(rev_voltage_sweep):
#     ps.ch1.vset = rev
#     delay(0.1)
#     ps.ch2.vset = rev - 5
# print(f"{rev:.01f}, {rev-5:.01f}")
ps.ch1.disable
ps.ch1.cfg_all = 0, 3

Voltage (V)   Current (uA)
0.0           -0.513921
0.1           -0.412302
0.2           -0.322057
0.3           -0.305736
0.4           -0.296431
0.5           -0.304555
0.6           -0.266596
0.7           -0.356915
0.8           -0.232993
0.9           -0.296948
1.0           -0.319399
1.1           -0.317590
1.2           -0.325825
1.3           -0.145851
1.4           -0.243481
1.5           -0.290708
1.6           -0.352779
1.7           -0.313343
1.8           -0.341702
1.9           -0.205890
2.0           -0.265636
2.1           -0.364597
2.2           -0.233143
2.3           -0.261871
2.4           -0.190716
2.5           -0.130048
2.6           -0.226571
2.7           -0.190311
2.8           -0.266967
2.9           -0.241339
3.0           -0.323758
3.1           -0.294217
3.2           -0.176647
3.3           -0.222655
3.4           -0.302857
3.5           -0.245105
3.6           -0.221363
3.7           -0.261796
3.8           -0.240305
3.9           -0.356546
4.0          

# VEXT_DRV pin test with VEXT pin

In [14]:
# absolute maximum rating

#-------------------#
max_rating = 36
item = "VEXT_DRV (VEXT_DRV-VEXT=5V, VOUT=4.0V)"
#-------------------# this method is for 

temp = list(np.arange(6, max_rating+10.2, 0.1))
voltage_sweep = [round(num, 1) for num in temp]

ps.ch1.cfg_all = 6, 0.01
ps.ch2.cfg_all = 1, 0.01


print(temp)
ic.VEXT_OFF_GATE_CTRL = 1
ic.VUSB_OFF_GATE_CTRL = 1
print("VUSB_GATE_OFF",ic.VUSB_OFF_GATE_CTRL)
ic.VEXT_OFF_GATE_CTRL = 1
ic.VUSB_OFF_GATE_CTRL = 1
print("VEXT_GATE_OFF",ic.VEXT_OFF_GATE_CTRL) # When Measuring VEXT/VUSB_DRV Gate Control Should be off

[np.float64(6.0), np.float64(6.1), np.float64(6.199999999999999), np.float64(6.299999999999999), np.float64(6.399999999999999), np.float64(6.499999999999998), np.float64(6.599999999999998), np.float64(6.6999999999999975), np.float64(6.799999999999997), np.float64(6.899999999999997), np.float64(6.9999999999999964), np.float64(7.099999999999996), np.float64(7.199999999999996), np.float64(7.299999999999995), np.float64(7.399999999999995), np.float64(7.499999999999995), np.float64(7.599999999999994), np.float64(7.699999999999994), np.float64(7.799999999999994), np.float64(7.899999999999993), np.float64(7.999999999999993), np.float64(8.099999999999993), np.float64(8.199999999999992), np.float64(8.299999999999992), np.float64(8.399999999999991), np.float64(8.499999999999991), np.float64(8.59999999999999), np.float64(8.69999999999999), np.float64(8.79999999999999), np.float64(8.89999999999999), np.float64(8.99999999999999), np.float64(9.099999999999989), np.float64(9.199999999999989), np.floa

In [15]:
chart.set_title = item

chart.set_single_plot
chart.set_xlabel = "Voltage (V)"
chart.set_y_main_label = "Current (uA)"

chart.add_main_item = "PIN"

chart.PIN.set_linewidth = 2
chart.PIN.set_line_dot

In [16]:
# for VEXT_DRV

ps.ch2.cfg_all = 1, 0.01
delay(0.5)
ps.ch1.cfg_all = 6, 0.01

ps.ch1.enable
ps.ch2.enable
delay(0.5)

for _ in range(3):
    dm1.current_100E_3

for set_volt in voltage_sweep:

    ps.ch2.vset = set_volt - 5
    delay(1)
    ps.ch1.vset = set_volt

    meas_i = dm1.current_100E_3 * 1e+6

    if meas_i > 2500:
        for _ in range(2):
            print("over-current")
            meas_i = dm1.current_100E_3 * 1e+6


    chart.PIN.set_data = set_volt, meas_i
    chart.output

    print(f"{set_volt:.01f}, {set_volt-5:.01f}, {meas_i:.06f}")

    if meas_i > 2500:
        break

chart.save_plot = chart.set_title

rev_temp = list(np.arange(6, set_volt, 0.1))
rev_voltage_sweep = [round(num, 1) for num in rev_temp]

for rev in reversed(rev_voltage_sweep):
    ps.ch1.vset = rev
    delay(0.1)
    ps.ch2.vset = rev - 5
    # print(f"{rev:.01f}, {rev-5:.01f}")

print(f"BV (V)           {set_volt:.01f}")
ps.ch1.disable
ps.ch1.cfg_all = 0, 0.01
ps.ch1.disable
ps.ch2.disable

6.0, 1.0, -0.235553
6.1, 1.1, -0.301059
6.2, 1.2, -0.199442
6.3, 1.3, -0.218865
6.4, 1.4, -0.141913
6.5, 1.5, -0.283999
6.6, 1.6, -0.219454
6.7, 1.7, -0.383992
6.8, 1.8, -0.326464
6.9, 1.9, -0.345666
7.0, 2.0, -0.289836
7.1, 2.1, -0.184377
7.2, 2.2, -0.294561
7.3, 2.3, -0.244711
7.4, 2.4, -0.175736
7.5, 2.5, -0.187552
7.6, 2.6, -0.234521
7.7, 2.7, -0.150701
7.8, 2.8, -0.221523
7.9, 2.9, -0.216352
8.0, 3.0, -0.099374
8.1, 3.1, -0.159120
8.2, 3.2, -0.196414
8.3, 3.3, -0.284148
8.4, 3.4, -0.204759
8.5, 3.5, -0.216576
8.6, 3.6, -0.336730
8.7, 3.7, -0.341900
8.8, 3.8, -0.264727
8.9, 3.9, -0.293012
9.0, 4.0, -0.096347
9.1, 4.1, -0.189027
9.2, 4.2, -0.106685
9.3, 4.3, 0.036806
9.4, 4.4, -0.171378
9.5, 4.5, -0.149444
9.6, 4.6, -0.158970
9.7, 4.7, -0.237104
9.8, 4.8, -0.189914
9.9, 4.9, -0.260220
10.0, 5.0, -0.214434
10.1, 5.1, -0.248776
10.2, 5.2, -0.258967
10.3, 5.3, -0.301283
10.4, 5.4, -0.176402
10.5, 5.5, -0.259115
10.6, 5.6, -0.117323
10.7, 5.7, -0.271079
10.8, 5.8, -0.320189
10.9, 5.9, -

# Lower Voltage Pins


In [6]:
# absolute maximum rating

#-------------------#
max_rating = 4
item = "VOUT"
#-------------------#

temp = list(np.arange(0 , max_rating+1.1, 0.1))
voltage_sweep = [round(num, 1) for num in temp]

ps.ch1.cfg_all = 0, 0.01
ps.ch2.cfg_all = 0, 0.01

chart.set_title = item

chart.set_single_plot
chart.set_xlabel = "Voltage (V)"
chart.set_y_main_label = "Current (uA)"

chart.add_main_item = "PIN"

chart.PIN.set_linewidth = 2
chart.PIN.set_line_dot


In [7]:
ps.ch1.enable
delay(0.5)
dm1.current_100E_3
for _ in range(3):
    dm1.current_100E_3

print("Voltage (V)   Current (uA)")

for set_volt in voltage_sweep:

    ps.ch1.vset = set_volt
    meas_i = dm1.current_100E_3 * 1e+6

    if meas_i > 5000:
        for _ in range(3):
            print("over-current, retest")
            meas_i = dm1.current_100E_3 * 1e+6
        
        if meas_i > 5000:
            print(f"{set_volt:.01f}           {meas_i:.06f}")
            break
    sleep(5)        

    chart.PIN.set_data = set_volt, meas_i
    chart.output

    print(f"{set_volt:.01f}           {meas_i:.06f}")

chart.save_plot = chart.set_title   
print(f"BV (V)           {set_volt:.01f}")
ps.ch1.disable
ps.ch1.cfg_all = 0, 0.01

Voltage (V)   Current (uA)
0.0           -0.317234
0.1           -0.258227
0.2           -0.077295
0.3           0.282356
0.4           3.077067
0.5           5.035498
0.6           12.884660
0.7           0.399926
0.8           0.363628
0.9           0.708915
1.0           1.303704
1.1           1.548517
1.2           1.878996
1.3           1.972269
1.4           2.325199
1.5           2.573335
1.6           2.629720
1.7           2.827896
1.8           3889.067000
1.9           4004.915000
2.0           801.267500
2.1           794.955100
2.2           139.446300
2.3           143.801500
2.4           148.219000
2.5           149.287700
2.6           149.832500
2.7           150.663600
2.8           150.519600
2.9           150.712800
3.0           150.783800
3.1           150.995800
3.2           151.151400
3.3           151.371300
3.4           151.534600
3.5           151.797000
3.6           151.650100
3.7           151.929800
3.8           152.161000
3.9           152.071000
4.0